# 50 · Gold analytics: OLAP queries for the business questions
**Owners: Ar (Q1, Q3, Q4) · M (Q2, Q5, Q6)** · Runs after: 40 (only on a validated load)

Each query answers one business question from P1 and is labelled with the OLAP operation it demonstrates.

| Query | Business question | OLAP operation |
|---|---|---|
| Q1 | BQ1 · Where is insured loss concentrated by hazard and state? | Roll-up (`GROUP BY ROLLUP`), ranking |
| Q1b | BQ1 · How much loss comes from the few largest events? | Window functions (running total) |
| Q2 | BQ2 · Is loss per event rising, nominal vs normalised? | Drill-down by financial year, moving average |
| Q3 | BQ3 · Which states are hit hardest relative to population? | Slice and dice across two facts |
| Q4 | BQ4 · When in the year do events happen? | Pivot (month × hazard) |
| Q5 | BQ5 · Is lower income linked to lower insured loss per person? | Cube (`GROUP BY CUBE`), correlation |
| Q6 | Data quality · How much loss would be unattributed without cleaning? | Before/after comparison |

Population and income are **non-additive across events**, so they are never summed: queries average them per state.

In [ ]:
%run ./00_common

In [ ]:
import matplotlib.pyplot as plt
SAVE_EVIDENCE = True
EVIDENCE_DIR = "./evidence/gold"
os.makedirs(EVIDENCE_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold", "axes.titlesize": 11})

F_, D, L, H, E, SY = (tname("gold", t) for t in ["fact_event_loss", "dim_date", "dim_location", "dim_hazard", "dim_event", "fact_state_year"])

def run(name, sql):
    df = spark.sql(sql)
    pdf = df.toPandas()
    if SAVE_EVIDENCE:
        pdf.to_csv(f"{EVIDENCE_DIR}/{name}.csv", index=False)
    display(df)
    return pdf

def savefig(name):
    plt.tight_layout()
    if SAVE_EVIDENCE:
        plt.savefig(f"{EVIDENCE_DIR}/{name}.png", bbox_inches="tight")
    plt.show()

### Q1 · Roll-up: normalised loss by hazard and state, with subtotals

In [ ]:
q1 = run("q1_rollup_hazard_state", f"""
SELECT hazard, state, events, normalised_loss_m, pct_of_total FROM (
SELECT grouping(h.hazard_group) AS g_hazard, grouping(l.state_code) AS g_state,
       CASE WHEN grouping(h.hazard_group) = 1 THEN 'All hazards' ELSE h.hazard_group END AS hazard,
       CASE WHEN grouping(l.state_code) = 1 THEN 'All states' ELSE l.state_code END AS state,
       count(*) AS events,
       round(sum(f.normalised_loss_2022_aud) / 1e6, 1) AS normalised_loss_m,
       round(100 * sum(f.normalised_loss_2022_aud) / (SELECT sum(normalised_loss_2022_aud) FROM {F_}), 1) AS pct_of_total
FROM {F_} f JOIN {H} h ON h.hazard_key = f.hazard_key JOIN {L} l ON l.location_key = f.location_key
GROUP BY ROLLUP (h.hazard_group, l.state_code))
ORDER BY g_hazard, hazard, g_state DESC, normalised_loss_m DESC""")

by_h = q1[(q1.state == "All states") & (q1.hazard != "All hazards")].sort_values("normalised_loss_m")
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(by_h.hazard, by_h.normalised_loss_m / 1000, color="#1f5f8b")
for y, (v, p) in enumerate(zip(by_h.normalised_loss_m / 1000, by_h.pct_of_total)):
    ax.text(v, y, f"  ${v:,.1f}bn ({p:.0f}%)", va="center", fontsize=8)
ax.set(title="Q1 · Normalised insured loss by hazard, 2011–2026", xlabel="AUD billions (2022 dollars)")
ax.set_xlim(0, by_h.normalised_loss_m.max() / 1000 * 1.35)
savefig("q1_loss_by_hazard")

### Q1b · Concentration: share of loss from the largest events

In [ ]:
q1b = run("q1b_concentration", f"""
WITH ranked AS (
  SELECT e.event_name, d.full_date AS start_date, h.hazard_group, l.state_code,
         f.normalised_loss_2022_aud AS loss,
         row_number() OVER (ORDER BY f.normalised_loss_2022_aud DESC) AS loss_rank,
         sum(f.normalised_loss_2022_aud) OVER (ORDER BY f.normalised_loss_2022_aud DESC ROWS UNBOUNDED PRECEDING)
           / sum(f.normalised_loss_2022_aud) OVER () AS cumulative_share
  FROM {F_} f JOIN {E} e ON e.event_key = f.event_key JOIN {D} d ON d.date_key = f.start_date_key
  JOIN {H} h ON h.hazard_key = f.hazard_key JOIN {L} l ON l.location_key = f.location_key
  WHERE f.normalised_loss_2022_aud IS NOT NULL)
SELECT loss_rank, event_name, start_date, hazard_group, state_code, round(loss / 1e6, 1) AS loss_m,
       round(100 * cumulative_share, 1) AS cumulative_pct
FROM ranked WHERE loss_rank <= 10 ORDER BY loss_rank""")

### Q2 · Drill-down: loss per event by Australian financial year

In [ ]:
q2 = run("q2_trend_by_fy", f"""
WITH fy AS (
  SELECT d.au_financial_year AS fy, count(*) AS events,
         count(f.normalised_loss_2022_aud) AS events_with_loss,
         sum(f.original_loss_aud) / 1e6 AS nominal_m, sum(f.normalised_loss_2022_aud) / 1e6 AS normalised_m
  FROM {F_} f JOIN {D} d ON d.date_key = f.start_date_key
  WHERE f.start_date_key > 0
  GROUP BY d.au_financial_year)
SELECT fy, events, events_with_loss, round(nominal_m, 1) AS nominal_loss_m, round(normalised_m, 1) AS normalised_loss_m,
       round(normalised_m / events_with_loss, 1) AS normalised_per_event_m,
       round(nominal_m / events_with_loss, 1) AS nominal_per_event_m,
       round(avg(normalised_m / events_with_loss) OVER (ORDER BY fy ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS per_event_3yr_avg_m
FROM fy ORDER BY fy""")
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(q2.fy, q2.normalised_loss_m / 1000, color="#9db7cf", label="total normalised loss ($bn, left)")
ax2 = ax.twinx(); ax2.plot(q2.fy, q2.per_event_3yr_avg_m, color="#c9822b", marker="o", label="loss per event, 3-yr avg ($m, right)")
ax.set(title="Q2 · Loss by financial year", xlabel="Australian financial year (ending June)", ylabel="$bn"); ax2.set_ylabel("$m per event")
ax2.spines["top"].set_visible(False)
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.88), fontsize=8, frameon=False)
savefig("q2_trend_by_fy")

### Drill-down example: the worst financial year by quarter and hazard

In [ ]:
worst_fy = int(q2.sort_values("normalised_loss_m", ascending=False).fy.iloc[0])
q2b = run("q2b_drilldown_worst_fy", f"""
SELECT d.au_financial_year AS fy, d.calendar_year, d.calendar_quarter, h.hazard_group, count(*) AS events,
       round(sum(f.normalised_loss_2022_aud) / 1e6, 1) AS normalised_loss_m
FROM {F_} f JOIN {D} d ON d.date_key = f.start_date_key JOIN {H} h ON h.hazard_key = f.hazard_key
WHERE d.au_financial_year = {worst_fy}
GROUP BY d.au_financial_year, d.calendar_year, d.calendar_quarter, h.hazard_group
ORDER BY d.calendar_year, d.calendar_quarter, normalised_loss_m DESC""")

### Q3 · Slice and dice across both facts: loss relative to population

In [ ]:
q3 = run("q3_state_per_capita", f"""
WITH ev AS (
  SELECT f.location_key, d.calendar_year, count(*) AS events, sum(f.normalised_loss_2022_aud) AS loss
  FROM {F_} f JOIN {D} d ON d.date_key = f.start_date_key
  WHERE f.location_key > 0
  GROUP BY f.location_key, d.calendar_year),
pop AS (SELECT location_key, avg(population) AS avg_population FROM {SY} GROUP BY location_key)
SELECT l.state_code, sum(ev.events) AS events, round(sum(ev.loss) / 1e6, 1) AS normalised_loss_m,
       round(max(pop.avg_population) / 1e6, 2) AS avg_population_m,
       round(sum(ev.events) / (max(pop.avg_population) / 1e6), 1) AS events_per_million_residents,
       round(sum(ev.loss) / max(pop.avg_population), 0) AS loss_per_resident_aud
FROM ev JOIN pop ON pop.location_key = ev.location_key JOIN {L} l ON l.location_key = ev.location_key
GROUP BY l.state_code ORDER BY loss_per_resident_aud DESC""")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
q3s = q3.sort_values("normalised_loss_m")
axes[0].barh(q3s.state_code, q3s.normalised_loss_m / 1000, color="#9db7cf"); axes[0].set(title="Total normalised loss ($bn)")
q3p = q3.sort_values("loss_per_resident_aud")
axes[1].barh(q3p.state_code, q3p.loss_per_resident_aud, color="#1f5f8b"); axes[1].set(title="Loss per resident ($)")
fig.suptitle("Q3 · Total loss vs loss per resident, by state (attributed events only)", fontweight="bold", fontsize=11)
savefig("q3_state_per_capita")

### Q4 · Pivot: events by month and hazard

In [ ]:
q4 = run("q4_pivot_month_hazard", f"""
SELECT * FROM (
  SELECT d.month_number, d.month_name, d.season, h.hazard_group
  FROM {F_} f JOIN {D} d ON d.date_key = f.start_date_key JOIN {H} h ON h.hazard_key = f.hazard_key
  WHERE f.start_date_key > 0)
PIVOT (count(*) FOR hazard_group IN ('Cyclone' AS cyclone, 'Storm' AS storm, 'Hail' AS hail, 'Flood' AS flood,
                                     'Bushfire' AS bushfire, 'Earthquake' AS earthquake))
ORDER BY month_number""")
q4 = q4.fillna(0)
haz = ["cyclone", "storm", "hail", "flood", "bushfire", "earthquake"]
fig, ax = plt.subplots(figsize=(9, 3.4))
bottom = None
colors = ["#12355b", "#1f5f8b", "#5b8db8", "#9db7cf", "#c9822b", "#e3b779"]
for h_, c in zip(haz, colors):
    ax.bar(q4.month_name.str[:3], q4[h_], bottom=bottom, label=h_.title(), color=c)
    bottom = q4[h_] if bottom is None else bottom + q4[h_]
ax.set(title="Q4 · Events by start month and hazard, 2011–2026", ylabel="events"); ax.legend(ncol=6, fontsize=8, frameon=False)
savefig("q4_events_by_month")
q4s = run("q4b_season", f"""
SELECT d.season, count(*) AS events, round(sum(f.normalised_loss_2022_aud) / 1e6, 1) AS normalised_loss_m,
       sum(f.total_claims) AS claims_recorded
FROM {F_} f JOIN {D} d ON d.date_key = f.start_date_key WHERE f.start_date_key > 0
GROUP BY d.season ORDER BY events DESC""")

### Q5 · Cube and correlation: loss per resident vs median income
Eight states is a tiny sample. Treat the correlation as a signal to investigate, not proof.

In [ ]:
q5 = run("q5_cube_state_family", f"""
SELECT state, hazard_family, events, normalised_loss_m FROM (
SELECT grouping(l.state_code) AS g_state, grouping(h.hazard_family) AS g_family,
       CASE WHEN grouping(l.state_code) = 1 THEN 'All' ELSE l.state_code END AS state,
       CASE WHEN grouping(h.hazard_family) = 1 THEN 'All' ELSE h.hazard_family END AS hazard_family,
       count(*) AS events, round(sum(f.normalised_loss_2022_aud) / 1e6, 1) AS normalised_loss_m
FROM {F_} f JOIN {L} l ON l.location_key = f.location_key JOIN {H} h ON h.hazard_key = f.hazard_key
WHERE f.location_key > 0
GROUP BY CUBE (l.state_code, h.hazard_family))
ORDER BY g_state, state, g_family, normalised_loss_m DESC""")

q5c = run("q5b_income_vs_loss", f"""
WITH s AS (
  SELECT l.state_code, avg(sy.median_income_aud) AS median_income, avg(sy.population) AS population
  FROM {SY} sy JOIN {L} l ON l.location_key = sy.location_key GROUP BY l.state_code),
loss AS (
  SELECT l.state_code, coalesce(sum(f.normalised_loss_2022_aud), 0) AS loss
  FROM {L} l LEFT JOIN {F_} f ON f.location_key = l.location_key WHERE l.location_key > 0 GROUP BY l.state_code)
SELECT s.state_code, round(s.median_income, 0) AS avg_median_income_aud, round(loss.loss / s.population, 0) AS loss_per_resident_aud
FROM s JOIN loss ON loss.state_code = s.state_code ORDER BY avg_median_income_aud""")
r = q5c[["avg_median_income_aud", "loss_per_resident_aud"]].astype(float).corr().iloc[0, 1]
print(f"Correlation between median income and loss per resident across {len(q5c)} states: r = {r:.2f}")
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.scatter(q5c.avg_median_income_aud / 1000, q5c.loss_per_resident_aud, color="#1f5f8b")
for _, row in q5c.iterrows():
    ax.annotate(row.state_code, (row.avg_median_income_aud / 1000, row.loss_per_resident_aud), xytext=(4, 3), textcoords="offset points", fontsize=8)
ax.set(title=f"Q5 · Income vs loss per resident (r = {r:.2f}, n = {len(q5c)})", xlabel="average median income ($k)", ylabel="loss per resident ($)")
savefig("q5_income_vs_loss")

### Q6 · Cleaning impact: loss that could not be placed in a state, before vs after
"Before" uses only the raw State column as a state code. "After" uses the cleaned location key.

In [ ]:
q6 = run("q6_cleaning_impact", f"""
WITH raw AS (
  SELECT s.normalised_loss_2022_aud AS loss,
         CASE WHEN upper(trim(s.state_raw)) IN ('NSW','VIC','QLD','SA','WA','TAS','NT','ACT') THEN 'placed' ELSE 'not placed' END AS before_cleaning,
         CASE WHEN s.state_code IS NOT NULL THEN 'placed' ELSE 'not placed' END AS after_cleaning
  FROM {tname('silver', 'ica_events')} s WHERE s.in_scope)
SELECT 'before cleaning' AS stage, count_if(before_cleaning = 'placed') AS events_placed, count(*) AS events,
       round(100 * sum(CASE WHEN before_cleaning = 'placed' THEN loss END) / sum(loss), 1) AS pct_loss_placed
FROM raw
UNION ALL
SELECT 'after cleaning', count_if(after_cleaning = 'placed'), count(*),
       round(100 * sum(CASE WHEN after_cleaning = 'placed' THEN loss END) / sum(loss), 1)
FROM raw""")